# Production Incremental Sync

This notebook implements a robust, long-term strategy for maintaining your Vector Database.
It handles:
1. **New Files**: Automatically detecting and adding them.
2. **Modified Files**: Detecting content changes via MD5 hash, deleting old versions, and adding new ones.
3. **Deleted Files**: Removing vectors for files that no longer exist on disk.

### State Tracking
This script maintains a file called `ingestion_state.json` in your DB directory. **Do not delete this file**, or the script will think everything is new and re-ingest the entire 105GB!

In [ ]:
import os
import glob
import json
import hashlib
import time
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Disable tokenizers parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# --- CONFIGURATION ---

# 1. Where represents the "Truth" (Your SSD Mirror)
SOURCE_DATA_PATH = "/mnt/e/WMS_selection"

# 2. Where the Database lives
DB_SAVE_PATH = "/mnt/e/chroma_db_wms"

# 3. The Ledger File (Tracks what we have already indexed)
STATE_FILE = os.path.join(DB_SAVE_PATH, "ingestion_state.json")

# 4. Model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# 5. Supported Extensions
EXTENSIONS = {'.md', '.txt', '.csv', '.py', '.json', '.html'}

In [ ]:
# --- HELPER FUNCTIONS ---

def calculate_md5(file_path):
    """Reads a file and returns its MD5 hash."""
    hash_md5 = hashlib.md5()
    try:
        with open(file_path, "rb") as f:
            # Read in chunks to avoid memory overflow on large files
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except Exception as e:
        print(f"Could not hash {file_path}: {e}")
        return None

def load_state():
    """Loads the ledger from disk."""
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r') as f:
            return json.load(f)
    return {} # Return empty dict if no state exists yet

def save_state(state):
    """Saves the ledger to disk."""
    # Ensure directory exists
    os.makedirs(os.path.dirname(STATE_FILE), exist_ok=True)
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)

In [ ]:
# --- 1. SCAN AND COMPARE ---

print("Loading previous state...")
previous_state = load_state()
current_state = {}
files_to_process = []
files_to_delete = []

print(f"Scanning {SOURCE_DATA_PATH} for changes... (This calculates hashes, so it takes a moment)")

all_files = glob.glob(os.path.join(SOURCE_DATA_PATH, "**"), recursive=True)
target_files = [f for f in all_files if os.path.splitext(f)[1].lower() in EXTENSIONS]

# Identify New and Modified Files
for file_path in target_files:
    file_hash = calculate_md5(file_path)
    if file_hash is None:
        continue
        
    current_state[file_path] = file_hash
    
    # Logic: Is it in the old state?
    if file_path not in previous_state:
        # CASE 1: New File
        files_to_process.append(("NEW", file_path))
    elif previous_state[file_path] != file_hash:
        # CASE 2: Modified File (Hash mismatch)
        files_to_process.append(("MODIFIED", file_path))

# Identify Deleted Files
# (Anything in previous_state that is NOT in current_state)
for old_path in previous_state:
    if old_path not in current_state:
        files_to_delete.append(old_path)

print(f"\n--- SCAN REPORT ---")
print(f"Total Files Found: {len(target_files)}")
print(f"Files to Add/Update: {len(files_to_process)}")
print(f"Files to Delete:     {len(files_to_delete)}")
print(f"Unchanged Files:     {len(target_files) - len(files_to_process)}")

if len(files_to_process) == 0 and len(files_to_delete) == 0:
    print("\nSystem is up to date. Nothing to do!")

In [ ]:
# --- 2. EXECUTE UPDATES ---

if len(files_to_process) > 0 or len(files_to_delete) > 0:
    print("Initializing Database...")
    embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)
    vectorstore = Chroma(
        persist_directory=DB_SAVE_PATH,
        embedding_function=embeddings
    )
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

    # -- A. HANDLE DELETIONS --
    if files_to_delete:
        print(f"\nProcessing {len(files_to_delete)} deletions...")
        for del_path in files_to_delete:
            print(f"  - Deleting vectors for: {os.path.basename(del_path)}")
            # Chroma allows deletion by metadata filter
            # Note: We assume 'source' metadata field was set correctly during ingestion
            try:
                vectorstore._collection.delete(where={"source": del_path})
            except Exception as e:
                print(f"    Error deleting {del_path}: {e}")

    # -- B. HANDLE ADDITIONS/UPDATES --
    if files_to_process:
        print(f"\nProcessing {len(files_to_process)} additions/updates...")
        
        for i, (action, file_path) in enumerate(files_to_process):
            try:
                # If Modified, we first delete the old version to avoid duplicates
                if action == "MODIFIED":
                    # print(f"  - Updating: {os.path.basename(file_path)}")
                    vectorstore._collection.delete(where={"source": file_path})
                
                # Load and Embed
                loader = TextLoader(file_path, encoding='utf-8', autodetect_encoding=True)
                docs = loader.load()
                chunks = text_splitter.split_documents(docs)
                
                if chunks:
                    vectorstore.add_documents(chunks)
                
                if i % 10 == 0:
                    print(f"  Processed {i+1}/{len(files_to_process)}...")
                    
            except Exception as e:
                print(f"  FAILED to process {file_path}: {e}")

    # -- C. COMMIT STATE --
    print("\nSaving new state to ledger...")
    # We save 'current_state' which represents exactly what is on disk right now
    save_state(current_state)
    
    print("SYNC COMPLETE! Database is now identical to SSD.")
else:
    pass # Nothing to do